# IndexTTS 2.5（现代 Web 工作台 + REST/WS API 服务）

- **支持显卡**：A100 (40GB / 80GB)、H100 (80GB)、RTX 6000 (96GB)
- **底层引擎**：官方原生 `index-tts`（零 ComfyUI 依赖，BF16 推理，含 QwenEmotion）
- **单页多合一能力**：零样本多语种克隆（`zh / en / ja / es / ar`）、8 维情绪向量、双音频音色/情绪分离、自然语言情绪控制、多音字标注与估算 SRT 字幕（非语音对齐）
- **访问方式**：默认使用 Cloudflare Quick Tunnel 临时公网域名（用完即关）；亦可在第 4 步切换「自有 Tunnel」从 Colab Secrets 读取 `CF_TUNNEL_TOKEN` 挂载固定域名
- **执行顺序**：`1. 环境检测` → `2. 安装 IndexTTS 2.5 引擎与应用服务包` → `3. 下载 IndexTTS 2.5 模型权重` → `4. 启动现代 Web 工作台与 API 服务`

| 入口路径 | 说明 |
| --- | --- |
| `GET /` | 现代前端可视化语音合成工作台（内置试听、录音、情绪面板与本地历史库） |
| `POST /api/v1/generate` | 异步提交语音合成任务（返回 `task_id`） |
| `GET /api/v1/tasks/{task_id}` | 轮询任务进度与音频/字幕结果（亦支持 `WS /api/v1/tasks/{task_id}/ws`） |
| `POST /api/v1/files` | 上传自定义参考音色或情绪参考音频 |
| `GET /docs` | Swagger / OpenAPI 在线交互文档 |

### 1. 环境检测
- **功能**：检测 GPU 型号、显存容量、BF16 支持、系统内存、磁盘空间及 PyTorch / CUDA 版本。
- **要求**：支持 `A100`、`H100`、`RTX 6000`，显存 40GB 及以上。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 1. 环境检测 {display-mode: "form"}
import os, shutil, subprocess, torch

MIN_VRAM_GIB = 35
REQUIREMENT = "40GB 及以上显存（A100 40GB / 80GB、H100 80GB、RTX 6000 96GB）"

if not torch.cuda.is_available():
    raise RuntimeError("未检测到 GPU。请在 Colab 菜单选择：代码执行程序 -> 更改运行时类型 -> A100 / H100 / RTX 6000 GPU。")

print(subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout)

gpu = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_gib = shutil.disk_usage("/content").free / 1024**3
bf16 = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu} | 显存: {vram_gib:.1f} GiB | 系统内存: {ram_gib:.1f} GiB | 磁盘可用: {disk_gib:.1f} GiB | BF16: {bf16}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

if not any(k in gpu.upper() for k in ("A100", "H100", "6000")) or vram_gib < MIN_VRAM_GIB:
    raise RuntimeError(f"当前显卡为 {gpu} ({vram_gib:.1f} GiB)，本笔记本需要 {REQUIREMENT}，请切换 runtime。")
if not bf16:
    raise RuntimeError("当前 GPU 不支持 BF16，请切换 runtime。")
print(f"[OK] GPU 环境正常：{gpu} ({vram_gib:.1f} GiB)")

### 2. 安装 IndexTTS 2.5 引擎与应用服务包
- **功能**：安装官方 `index-tts` 推理引擎、FastAPI 依赖、`cloudflared` `2026.10.6`，并拉取校验 GitHub Release 发布的 `app-indextts-2.5.tar.gz`（`v0.2.0`，内含编译好的现代前端 `dist` 与 API 服务）。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 2. 安装 IndexTTS 2.5 引擎与应用服务包 {display-mode: "form"}
import hashlib, shutil, subprocess, sys, tarfile, tempfile, urllib.request
from pathlib import Path

INDEXTTS_DIR = Path("/content/index-tts")
INDEXTTS_REPO = "https://github.com/index-tts/index-tts"
INDEXTTS_REVISION = "d9e41aac89fd00b3d71497fddb287b7f24613712"

APP_DIR = Path("/content/app")
RELEASE_REPO = "boltguo/colab"
RELEASE_TAG = "v0.2.0"
APP_ARCHIVE = "app-indextts-2.5.tar.gz"
APP_SHA256 = "a11cc6c4a863b3c947dfeea933f3bdec6dbe4df60c92d0fad194ddfc6c37ea29"
CLOUDFLARED_VERSION = "2026.10.6"

RUNTIME_PACKAGES = [
    "transformers==4.52.1",
    "tokenizers>=0.21.0",
    "accelerate>=1.8.0",
    "safetensors>=0.4.3",
    "huggingface_hub>=0.23.0",
    "hf_xet",
    "librosa>=0.10.2",
    "soundfile>=0.10.0",
    "scipy>=1.10.0",
    "omegaconf>=2.3.0",
    "einops>=0.8.1",
    "sentencepiece>=0.2.1",
    "tiktoken>=0.5.0",
    "openai-whisper>=20231117",
    "modelscope>=1.27.0",
    "munch>=4.0.0",
    "json5>=0.9.16",
    "textstat>=0.7.10",
    "jieba>=0.42.1",
    "cn2an>=0.5.22",
    "g2p-en>=2.1.0",
    "ffmpeg-python>=0.2.0",
    "descript-audiotools>=0.7.2",
    "PyYAML>=6.0",
    "wetext>=0.0.9",
    "fugashi>=1.2.0",
    "unidic-lite>=1.0.0",
    "fastapi>=0.115.0,<1.0.0",
    "uvicorn[standard]>=0.30.0,<1.0.0",
    "python-multipart>=0.0.9,<1.0.0",
    "pydantic>=2.7.0,<3.0.0",
]

def run(args, cwd=None):
    subprocess.run(args, cwd=cwd, check=True)

def git(path, *args):
    return subprocess.check_output(["git", *args], cwd=path, text=True).strip()

def ensure_not_symlink(path, label):
    for p in [path, *path.parents]:
        if p == Path("/content"):
            break
        if p.is_symlink():
            raise RuntimeError(f"{label} 路径包含软链接，请使用新的 runtime：{p}")

def checkout_repo(path, repo_url, revision, label, allow_local_changes=False):
    ensure_not_symlink(path, label)
    fresh = not path.exists()
    if fresh:
        path.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--depth", "1", "--no-checkout", repo_url + ".git", str(path)])
    if not (path / ".git").is_dir():
        raise RuntimeError(f"{label} 目录不是 Git 仓库，请使用新的 runtime。")
    if git(path, "remote", "get-url", "origin").rstrip("/").removesuffix(".git") != repo_url:
        raise RuntimeError(f"现有 {label} 不是指定仓库，请使用新的 runtime。")
    if not fresh and not allow_local_changes and git(path, "status", "--porcelain"):
        raise RuntimeError(f"现有 {label} 有未提交修改，请使用新的 runtime。")
    if fresh or git(path, "rev-parse", "HEAD") != revision:
        run(["git", "fetch", "--depth", "1", "origin", revision], cwd=path)
        run(["git", "checkout", "--force", "--detach", revision], cwd=path)

def install_release_app():
    ensure_not_symlink(APP_DIR, "应用目录")
    stamp_file = APP_DIR / ".release_sha256"
    if (
        stamp_file.is_file()
        and not stamp_file.is_symlink()
        and stamp_file.read_text(encoding="utf-8").strip() == f"{RELEASE_TAG}:{APP_SHA256}"
        and (APP_DIR / "backend/run_server.py").is_file()
    ):
        print(f"应用包已是最新：{APP_ARCHIVE} ({RELEASE_TAG})")
        return

    local_archive = Path(f"/content/{APP_ARCHIVE}")
    with tempfile.NamedTemporaryFile(suffix=".tar.gz", delete=False) as tmp:
        tmp_path = Path(tmp.name)
    try:
        if local_archive.is_file() and not local_archive.is_symlink():
            print(f"检测到本地预置应用包：{local_archive}")
            payload = local_archive.read_bytes()
        else:
            url = f"https://github.com/{RELEASE_REPO}/releases/download/{RELEASE_TAG}/{APP_ARCHIVE}"
            print(f"正在拉取应用包：{url}")
            req = urllib.request.Request(url, headers={"User-Agent": "colab-release-fetcher"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                payload = resp.read()
        actual_sha = hashlib.sha256(payload).hexdigest()
        if actual_sha != APP_SHA256:
            raise RuntimeError(
                f"应用包 SHA256 校验失败！预期 {APP_SHA256}，实际 {actual_sha}"
            )
        tmp_path.write_bytes(payload)

        staging = APP_DIR.with_name("app.staging")
        if staging.is_symlink():
            raise RuntimeError("staging 目录为软链接，拒绝写入。")
        if staging.exists():
            shutil.rmtree(staging)
        staging.mkdir(parents=True, exist_ok=True)
        staging_root = staging.resolve()

        with tarfile.open(tmp_path, "r:gz") as tf:
            for member in tf.getmembers():
                if member.issym() or member.islnk():
                    raise RuntimeError(f"拒绝解压含链接的归档条目：{member.name}")
                target = (staging / member.name).resolve()
                if target != staging_root and not str(target).startswith(str(staging_root) + "/"):
                    raise RuntimeError(f"归档路径越界：{member.name}")
            tf.extractall(staging)

        (staging / ".release_sha256").write_text(f"{RELEASE_TAG}:{APP_SHA256}\n", encoding="utf-8")
        if APP_DIR.exists():
            shutil.rmtree(APP_DIR)
        staging.replace(APP_DIR)
    finally:
        if tmp_path.exists():
            tmp_path.unlink()

checkout_repo(INDEXTTS_DIR, INDEXTTS_REPO, INDEXTTS_REVISION, "index-tts", allow_local_changes=True)

# 让 Linux 也走预编译的 wetext 文本规范化，避免现场编译 pynini
front_py = INDEXTTS_DIR / "indextts/utils/front.py"
front_py.write_text(
    front_py.read_text(encoding="utf-8").replace('if platform.system() != "Linux":', "if True:"),
    encoding="utf-8",
)

run([sys.executable, "-m", "pip", "install", "-q", *RUNTIME_PACKAGES])
# descript-audiotools 声明了过旧的 protobuf<3.20 约束，安装后恢复现代 protobuf 以兼容 Colab Python 3.13 / TF
run([sys.executable, "-m", "pip", "install", "-q", "-U", "protobuf>=5.28.0,<7.0.0"])
install_release_app()

cloudflared_bin = Path("/usr/local/bin/cloudflared")
try:
    installed = subprocess.run([str(cloudflared_bin), "--version"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout
except OSError:
    installed = ""
if CLOUDFLARED_VERSION not in installed.split():
    download = cloudflared_bin.with_name("cloudflared.download")
    urllib.request.urlretrieve(
        f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64",
        str(download),
    )
    download.chmod(0o755)
    download.replace(cloudflared_bin)

print(f"index-tts: {INDEXTTS_REVISION[:12]} | app: {RELEASE_TAG} ({APP_SHA256[:12]}) | cloudflared: {CLOUDFLARED_VERSION}")
print("[OK] IndexTTS 2.5 引擎与 Web/API 应用包安装完成（零 ComfyUI 依赖）。")

### 3. 下载 IndexTTS 2.5 模型权重
- **功能**：下载 IndexTTS-2.5 权重（含 QwenEmotion）与辅助编码器 / 声码器（约 5.1 GB）至 `/content/models/IndexTTS-2.5`，并预缓存辅助特征提取与声码器模型。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 3. 下载 IndexTTS 2.5 模型权重 {display-mode: "form"}
import os, shutil, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
from huggingface_hub import hf_hub_download

MODEL_DIR = Path("/content/models/IndexTTS-2.5")
INDEXTTS_DIR = Path("/content/index-tts")
INDEXTTS = ("IndexTeam/IndexTTS-2.5", "c39ce5ba981572cb187443877ff559dfb246ce63", "")
MODEL_FILES = [
    (*INDEXTTS, "config.yaml", 2860),
    (*INDEXTTS, "gpt.pth", 3259599833),
    (*INDEXTTS, "s2mel.pth", 414908601),
    (*INDEXTTS, "codec.pth", 607290935),
    (*INDEXTTS, "wav2vec2bert_stats.pt", 9343),
    (*INDEXTTS, "feat1.pt", 57170),
    (*INDEXTTS, "feat2.pt", 374866),
    (*INDEXTTS, "multilingual_zh_ja_yue_char_del.tiktoken", 907395),
    (*INDEXTTS, "qwen0.6bemo4-merge/Modelfile", 360),
    (*INDEXTTS, "qwen0.6bemo4-merge/added_tokens.json", 707),
    (*INDEXTTS, "qwen0.6bemo4-merge/chat_template.jinja", 550),
    (*INDEXTTS, "qwen0.6bemo4-merge/config.json", 727),
    (*INDEXTTS, "qwen0.6bemo4-merge/generation_config.json", 117),
    (*INDEXTTS, "qwen0.6bemo4-merge/merges.txt", 1671853),
    (*INDEXTTS, "qwen0.6bemo4-merge/model.safetensors", 1192135096),
    (*INDEXTTS, "qwen0.6bemo4-merge/special_tokens_map.json", 616),
    (*INDEXTTS, "qwen0.6bemo4-merge/tokenizer.json", 11422654),
    (*INDEXTTS, "qwen0.6bemo4-merge/tokenizer_config.json", 5433),
    (*INDEXTTS, "qwen0.6bemo4-merge/vocab.json", 2776833),
]

def local_directory(path):
    path = Path(path)
    for parent in [path, *path.parents]:
        if parent == Path("/content"):
            break
        if parent.is_symlink():
            raise RuntimeError(f"目录是软链接，请使用新的 runtime：{parent}")
    path.mkdir(parents=True, exist_ok=True)

def model_ready(target, size):
    return target.is_file() and not target.is_symlink() and target.stat().st_size == size

def download_model(repo, revision, _folder, file, size):
    target = MODEL_DIR / file
    local_directory(target.parent)
    started = time.time()
    result = Path(
        hf_hub_download(
            repo_id=repo,
            filename=file,
            revision=revision,
            local_dir=str(MODEL_DIR),
            force_download=target.exists(),
            token=False,
        )
    )
    if result.resolve() != target.resolve() or not model_ready(target, size):
        raise RuntimeError(f"模型下载不完整：{target}")
    print(f"已下载：{file}（{size / 1024**3:.2f} GB，{time.time() - started:.0f} 秒）")

def download_models():
    local_directory(MODEL_DIR)
    pending = []
    for item in MODEL_FILES:
        target = MODEL_DIR / item[3]
        if target.is_symlink():
            raise RuntimeError(f"模型文件是软链接：{target}")
        if model_ready(target, item[4]):
            print("已存在：", item[3])
        else:
            pending.append(item)
    if pending:
        missing = sum(item[4] for item in pending)
        free = shutil.disk_usage(MODEL_DIR).free
        if free < missing + 2 * 1024**3:
            raise RuntimeError(
                f"磁盘空间不足：还需 {missing / 1024**3:.1f} GB，可用 {free / 1024**3:.1f} GB。"
            )
        started = time.time()
        pool = ThreadPoolExecutor(max_workers=3)
        try:
            futures = [pool.submit(download_model, *item) for item in sorted(pending, key=lambda x: -x[4])]
            for future in as_completed(futures):
                future.result()
        finally:
            pool.shutdown(wait=False, cancel_futures=True)
        elapsed = time.time() - started
        print(
            f"主权重下载完成：{missing / 1024**3:.1f} GB，用时 {elapsed:.0f} 秒（{missing / 1024**2 / max(elapsed, 1):.0f} MB/s）"
        )

    if str(INDEXTTS_DIR) not in sys.path:
        sys.path.insert(0, str(INDEXTTS_DIR))
    from indextts.utils.model_download import ensure_models_available
    ensure_models_available(str(MODEL_DIR))

download_models()
print("[OK] IndexTTS 2.5 全部主模型与辅助声码器准备完成。")

### 4. 启动现代 Web 工作台与 API 服务
- **功能**：启动 FastAPI 服务（`127.0.0.1:8188`，同时承载 `/` 现代前端工作台、`/api/v1/*` 业务接口与 `/docs` OpenAPI 文档），并挂载 Cloudflare Tunnel。
- **隧道选择**：`TUNNEL_MODE` 默认选择 `Quick Tunnel (临时链接)`（免费公网临时域名，实例关闭即失效）；也可切换为 `自有 Tunnel`，从 Colab 左侧 **🔑 Secrets** 读取 `CF_TUNNEL_TOKEN` 挂载固定域名。
- **访问密码**：默认开启 `REQUIRE_API_KEY`。在下方 `ACCESS_PASSWORD` 填写密码，或留空并在 Secrets 配置 `APP_API_KEY`。打开页面后先输入密码；API 使用同一密码作为 Bearer Key。共享笔记本时建议使用 Secrets。
- **使用**：直接运行，点击输出的公网链接即可在网页操作或直接复制对应链接调用 API。

In [ ]:
# @title 4. 启动现代 Web 工作台与 API 服务 {display-mode: "form"}
TUNNEL_MODE = "Quick Tunnel (临时链接)"  # @param ["Quick Tunnel (临时链接)", "自有 Tunnel (读取 Secrets 中 CF_TUNNEL_TOKEN)"]
REQUIRE_API_KEY = True  # @param {type:"boolean"}
ACCESS_PASSWORD = ""  # @param {type:"string"}
USE_CF_TUNNEL_TOKEN = TUNNEL_MODE.startswith("自有")
import hashlib, json, os, re, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

FEATURE_NAME = "indextts-2.5"
APP_DIR = Path("/content/app")
DATA_DIR = Path(f"/content/app_data/{FEATURE_NAME}")
PORT = 8188
BASE = f"http://127.0.0.1:{PORT}"
LOG = Path(f"/content/{FEATURE_NAME}-web.log")
LAUNCH_CONFIG = Path(f"/content/{FEATURE_NAME}.launch.json")
PROCESS_PATTERN = r"backend/run_server\.py|backend\.run_server"

CF_BIN = Path("/usr/local/bin/cloudflared")
CF_LOG = Path("/content/cloudflared.log")

def read_text(path):
    return path.read_text(errors="replace") if path.exists() else ""

def tail(path, size=8000):
    return read_text(path)[-size:]

def rotate_log(path):
    if path.is_file() and path.stat().st_size:
        path.replace(path.with_name(path.name + ".1"))

def process_running(pattern):
    return subprocess.run(["pgrep", "-f", pattern], stdout=subprocess.DEVNULL).returncode == 0

def stop_processes(pattern, timeout):
    subprocess.run(["pkill", "-f", pattern], check=False)
    deadline = time.time() + timeout
    while process_running(pattern) and time.time() < deadline:
        time.sleep(2)
    if process_running(pattern):
        subprocess.run(["pkill", "-9", "-f", pattern], check=False)
        time.sleep(2)

def resolve_cf_token():
    t = ""
    try:
        from google.colab import userdata
        t = (userdata.get("CF_TUNNEL_TOKEN") or "").strip()
    except Exception:
        t = ""
    if not t:
        t = os.environ.get("CF_TUNNEL_TOKEN", "").strip()
    return t.split()[-1] if t else ""

def start_cf_tunnel(token=""):
    if not CF_BIN.exists():
        raise RuntimeError("未找到 cloudflared，请先运行第 2 步。")
    stop_processes("cloudflared tunnel", 10)
    rotate_log(CF_LOG)
    cmd = [
        str(CF_BIN),
        "tunnel",
        "--no-autoupdate",
        "--protocol", "http2",
        "--edge-ip-version", "4",
    ]
    if token:
        cmd.extend(["run", "--url", BASE, "--token", token])
    else:
        cmd.extend(["--url", BASE])
    with CF_LOG.open("w") as cf_log:
        cf_proc = subprocess.Popen(
            cmd,
            stdin=subprocess.DEVNULL,
            stdout=cf_log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    deadline = time.time() + (30 if token else 60)
    while time.time() < deadline:
        if cf_proc.poll() is not None:
            print(tail(CF_LOG, 4000))
            if token:
                raise RuntimeError("Cloudflare Tunnel 启动失败，请检查 CF_TUNNEL_TOKEN 是否正确。")
            raise RuntimeError("Cloudflare 临时隧道启动失败，请查看上方日志。")
        log_text = read_text(CF_LOG)
        if not token:
            match = re.search(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
            if match and "Registered tunnel connection" in log_text:
                url = match.group(0)
                print(f"[OK] Cloudflare 临时隧道已连接 (HTTP/2): {url} -> {BASE}")
                return url
        elif "Registered tunnel connection" in log_text:
            if "Updated to new configuration" not in log_text:
                time.sleep(1)
                log_text = read_text(CF_LOG)
            configs = [line for line in log_text.splitlines() if "Updated to new configuration" in line]
            config = configs[-1].replace('\\"', '"') if configs else ""
            routes = re.findall(r'"hostname"\s*:\s*"([^"]+)"(?:(?!"hostname").)*?"service"\s*:\s*"([^"]+)"', config)
            for host, service in routes:
                print(f"[OK] Cloudflare Tunnel 已连接 (HTTP/2): https://{host} -> {service}")
            hosts = [host for host, service in routes if f":{PORT}" in service]
            if not routes:
                print("[OK] Cloudflare Tunnel 已连接 (HTTP/2)，请通过你在 Cloudflare 绑定的域名访问。")
            elif not hosts:
                print(f"[WARN] 没有域名转发到本服务的 {PORT} 端口！")
                print(f"       请前往 Cloudflare Zero Trust -> Networks -> Tunnels -> Public Hostname，将 Service URL 改为 {BASE}")
            return f"https://{hosts[0]}" if hosts else "https://<你的CF域名>"
        time.sleep(1)
    if not token:
        match = re.search(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com", read_text(CF_LOG))
        if match:
            url = match.group(0)
            print(f"[OK] Cloudflare 临时隧道已创建: {url} -> {BASE}")
            return url
        print(tail(CF_LOG, 4000))
        raise RuntimeError(f"Cloudflare 临时隧道 60 秒内未返回地址，详见 {CF_LOG}。")
    print(f"[WARN] Cloudflare Tunnel 30 秒内未确认连接，仍在后台运行，详见 {CF_LOG}。")
    return "https://<你的CF域名>"

def check_public_url(url, route):
    if "<" in url:
        return
    error = None
    deadline = time.time() + 60
    while time.time() < deadline:
        try:
            request = urllib.request.Request(url + route, headers={"User-Agent": "colab-tunnel-check"})
            with urllib.request.urlopen(request, timeout=10):
                print(f"[OK] 公网访问正常：{url}")
                return
        except Exception as e:
            error = e
        time.sleep(3)
    print(f"[WARN] 公网地址 60 秒内无法访问：{url}{route}（{error}）")

cf_token = resolve_cf_token() if USE_CF_TUNNEL_TOKEN else ""
if USE_CF_TUNNEL_TOKEN and not cf_token:
    raise RuntimeError("已勾选 USE_CF_TUNNEL_TOKEN，但未检测到 CF_TUNNEL_TOKEN；请在 Colab 左侧 🔑 Secrets 中添加 CF_TUNNEL_TOKEN 并开启访问权限，或取消勾选改用临时隧道。")

if APP_DIR.is_symlink() or not (APP_DIR / "backend/run_server.py").is_file():
    raise RuntimeError("未找到应用服务目录 /content/app，请先运行第 2 步。")

if FEATURE_NAME == "indextts-2.5":
    missing_media_tools = [name for name in ("ffmpeg", "ffprobe") if not shutil.which(name)]
    if missing_media_tools:
        raise RuntimeError(f"缺少媒体处理工具：{', '.join(missing_media_tools)}。请先执行 !apt-get -qq update && apt-get -qq install -y ffmpeg，再运行第 4 步。")

def resolve_app_api_key():
    if ACCESS_PASSWORD.strip():
        return ACCESS_PASSWORD.strip()
    key = ""
    try:
        from google.colab import userdata
        key = (userdata.get("APP_API_KEY") or "").strip()
    except Exception:
        key = ""
    if not key:
        key = os.environ.get("APP_API_KEY", "").strip()
    return key

app_api_key = resolve_app_api_key() if REQUIRE_API_KEY else ""
if REQUIRE_API_KEY and not app_api_key:
    raise RuntimeError(
        "请在 ACCESS_PASSWORD 填写访问密码，或在 Colab 左侧 🔑 Secrets 配置 APP_API_KEY。"
    )

def ready(timeout=5):
    try:
        with urllib.request.urlopen(BASE + "/api/v1/health", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

env_overrides = {
    "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
    "COLAB_FEATURE": FEATURE_NAME,
    "COLAB_HOST": "127.0.0.1",
    "COLAB_PORT": str(PORT),
    "COLAB_DATA_DIR": str(DATA_DIR),
    "COLAB_DIST_DIR": str(APP_DIR / "dist" / FEATURE_NAME),
    "COLAB_API_KEY": app_api_key,
    "INDEXTTS_SRC_DIR": "/content/index-tts",
    "INDEXTTS_MODEL_DIR": "/content/models/IndexTTS-2.5",
}

cmd = [
    sys.executable,
    str(APP_DIR / "backend/run_server.py"),
    "--feature",
    FEATURE_NAME,
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
    "--dist-dir",
    str(APP_DIR / "dist" / FEATURE_NAME),
]

launch_fingerprint = json.dumps({
    "cmd": cmd,
    "release": read_text(APP_DIR / ".release_sha256").strip(),
    "auth_sha256": hashlib.sha256(app_api_key.encode("utf-8")).hexdigest(),
    "env": {key: value for key, value in env_overrides.items() if key != "COLAB_API_KEY"},
}, sort_keys=True)
running = ready() or (process_running(PROCESS_PATTERN) and ready(timeout=15))
if running and read_text(LAUNCH_CONFIG) != launch_fingerprint:
    print("服务配置已变更，正在重启应用服务...")
    stop_processes(PROCESS_PATTERN, 30)
    running = False

if running:
    print("[OK] Web 与 API 服务已经在运行。")
else:
    stop_processes(PROCESS_PATTERN, 30)
    rotate_log(LOG)
    child_env = {**os.environ, **env_overrides}
    with LOG.open("w") as log:
        process = subprocess.Popen(
            cmd,
            cwd=APP_DIR,
            env=child_env,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    LAUNCH_CONFIG.write_text(launch_fingerprint, encoding="utf-8")
    print("正在启动 FastAPI 服务与静态前端工作台...")
    deadline = time.time() + 120
    while time.time() < deadline:
        if process.poll() is not None:
            print(tail(LOG))
            raise RuntimeError(f"应用服务启动失败（退出码 {process.returncode}），详见上方日志。")
        if ready():
            print("[OK] Web 与 API 服务启动成功。")
            break
        time.sleep(1.5)
    else:
        print(tail(LOG))
        raise TimeoutError("应用服务 2 分钟内未就绪，请查看上方日志。")

host_url = start_cf_tunnel(cf_token)
tunnel_mode_key = "custom" if cf_token else "quick"
if not DATA_DIR.is_symlink():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    tunnel_info_path = DATA_DIR / "tunnel_info.json"
    if not tunnel_info_path.is_symlink():
        tunnel_info_path.write_text(
            json.dumps({"public_url": host_url, "tunnel_mode": tunnel_mode_key}, ensure_ascii=False),
            encoding="utf-8",
        )
check_public_url(host_url, "/api/v1/health")

sample_body = json.dumps(
    {"text": "你好，欢迎调用 IndexTTS 2.5 API 服务！", "speaker_audio": "preset:ref_audio", "language": "zh"},
    ensure_ascii=False,
)
auth_note = ' -H "Authorization: Bearer $APP_API_KEY"' if app_api_key else ""
tunnel_tip = (
    "⚠️ 提示：当前使用 Cloudflare Quick Tunnel 临时公网域名，实例关闭或重连隧道后地址即失效；如需固定域名请在上方选择「自有 Tunnel」。"
    if tunnel_mode_key == "quick"
    else "✅ 当前使用自有 Cloudflare Tunnel 固定域名。"
)
print(f"""
=== 访问入口 ===
🌐 现代 Web 操作台 : {host_url}
📖 在线 API 文档   : {host_url}/docs
🩺 系统与显存状态  : {host_url}/api/v1/system
{tunnel_tip}

=== 快速 API 调用示例 ===
curl -X POST "{host_url}/api/v1/generate" \\
  -H "Content-Type: application/json"{auth_note} \\
  -d '{sample_body}'
""")